In [1]:
import pandas as pd

------

In [2]:
ATC_mapping = {
    "A": "Alimentary tract and metabolism",
    "B": "Blood and blood-forming organs",
    "C": "Cardiovascular system",
    "D": "Dermatologicals",
    "G": "Genito-urinary system and sex hormones",
    "H": "Systemic hormonal preparations, excluding sex hormones and insulins",
    "J": "Antiinfectives for systemic use",
    "L": "Antineoplastic and immunomodulating agents",
    "M": "Musculoskeletal system",
    "N": "Nervous system",
    "P": "Antiparasitic products, insecticides and repellents",
    "R": "Respiratory system",
    "S": "Sensory organs",
    "V": "Various"}

def apply_atc_mapping(atc_code):
    """Map ATC code to its description."""
    if atc_code in ATC_mapping:
        return ATC_mapping[atc_code]
    else:
        return "Unknown"

In [3]:
def txt_into_df(txt_file, columns):
    data = []
    with open(txt_file, "r", encoding="utf-8") as file:
        file_list = file.readlines()
        for line in file_list:
            line = line.strip().split("\",\"")
            line = [item.replace('"', '') for item in line]
            data.append(line)

    df = pd.DataFrame(data, columns=columns)
    df = df.drop(columns=["_"])
    return df

In [69]:
drug = "./../data/HealthCanada/allfiles/drug.txt"
ingred = "./../data/HealthCanada/allfiles/ingred.txt"
form = "./../data/HealthCanada/allfiles/form.txt"
route = "./../data/HealthCanada/allfiles/route.txt"
status = "./../data/HealthCanada/allfiles/status.txt"
ther = "./../data/HealthCanada/allfiles/ther.txt"
comp = "./../data/HealthCanada/allfiles/comp.txt"

# drug

In [22]:

columns_drug=["DRUG_CODE", "_", "CLASS", "DRUG_ID", "BRAND_NAME", "_", "_", "_", "_", "_", "_", "_", "_", "_"]
df_drug = txt_into_df(drug, columns_drug)
print(len(df_drug) == df_drug["DRUG_CODE"].nunique())
df_drug.head()

True


,DRUG_CODE,CLASS,DRUG_ID,BRAND_NAME
0,9,Human,00015741,TAPAZOLE
1,15,Human,00015229,AVENTYL
2,16,Human,00015237,AVENTYL
3,57,Human,00050520,MINERALE LEGERE HUILE
4,68,Human,00050466,METHYLENE BLEU LIQ 1%


In [42]:
# classes to keep
classes_to_keep = ["Human", "Radiopharmaceutical"]
df_drug_filtered = df_drug[df_drug["CLASS"].isin(classes_to_keep)]
df_drug_filtered["CLASS"].value_counts()

CLASS
Human                  11462
Radiopharmaceutical       61
Name: count, dtype: int64

In [43]:
print(df_drug_filtered["DRUG_ID"].nunique() == df_drug_filtered["DRUG_CODE"].nunique())
df_drug_filtered["DRUG_ID"].nunique()
df_drug_filtered["DRUG_CODE"].nunique()

True


11523

# ingred

In [38]:
columns_ingred = ["DRUG_CODE", "_", "INGREDIENT", "_", "_", "_", "_", "_", "_", "_", "_", "_", "_", "_", "_"]
df_ingred = txt_into_df(ingred, columns_ingred)
print(len(df_ingred) == df_ingred["DRUG_CODE"].nunique())
print(len(df_ingred))
df_ingred.sort_values(by="DRUG_CODE", inplace=True)
df_ingred.head(10)

False
19825


,DRUG_CODE,INGREDIENT
18908,,
1189,10000,LINCOMYCIN (LINCOMYCIN HYDROCHLORIDE)
16547,100000,HOMOSALATE
16548,100000,AVOBENZONE
16550,100000,OCTOCRYLENE
16549,100000,OCTISALATE
16551,100001,AVOBENZONE
16552,100001,OCTOCRYLENE
16554,100001,OCTISALATE
16553,100001,HOMOSALATE


In [39]:
# Check for duplicates first
print(f"Total rows: {len(df_ingred)}")
print(f"Unique DRUG_CODEs: {df_ingred['DRUG_CODE'].nunique()}")
print(f"Has duplicates: {len(df_ingred) != df_ingred['DRUG_CODE'].nunique()}")

# Group by DRUG_CODE and combine ingredients with semicolon
df_ingred_combined = df_ingred.groupby('DRUG_CODE')['INGREDIENT'].apply(
    lambda x: '; '.join(x.astype(str).unique())
).reset_index()

print(f"After combining - Total rows: {len(df_ingred_combined)}")
print(f"After combining - Unique DRUG_CODEs: {df_ingred_combined['DRUG_CODE'].nunique()}")

# Show some examples of combined ingredients
df_ingred_combined.sort_values(by="DRUG_CODE", inplace=True)
df_ingred_combined.head(10)


Total rows: 19825
Unique DRUG_CODEs: 13482
Has duplicates: True
After combining - Total rows: 13482
After combining - Unique DRUG_CODEs: 13482


,DRUG_CODE,INGREDIENT
0,,
1,10000,LINCOMYCIN (LINCOMYCIN HYDROCHLORIDE)
2,100000,HOMOSALATE; AVOBENZONE; OCTOCRYLENE; OCTISALATE
3,100001,AVOBENZONE; OCTOCRYLENE; OCTISALATE; HOMOSALATE
4,100002,EPTINEZUMAB
5,10001,MELENGESTROL ACETATE
6,100012,KETOROLAC TROMETHAMINE
7,100015,CETYLPYRIDINIUM CHLORIDE
8,100018,BENZALKONIUM CHLORIDE
9,100019,ESCITALOPRAM (ESCITALOPRAM OXALATE)


# Merge

In [50]:
# merge drug and ingredient data
df_merged_a = pd.merge(df_drug_filtered, df_ingred_combined, on="DRUG_CODE", how="inner")
print(f"After merging - Total rows: {len(df_merged_a)}")
df_merged_a.head(10)

After merging - Total rows: 11523


,DRUG_CODE,CLASS,DRUG_ID,BRAND_NAME,INGREDIENT
0,9,Human,00015741,TAPAZOLE,METHIMAZOLE
1,15,Human,00015229,AVENTYL,NORTRIPTYLINE (NORTRIPTYLINE HYDROCHLORIDE)
2,16,Human,00015237,AVENTYL,NORTRIPTYLINE (NORTRIPTYLINE HYDROCHLORIDE)
3,57,Human,00050520,MINERALE LEGERE HUILE,MINERAL OIL LIGHT
4,68,Human,00050466,METHYLENE BLEU LIQ 1%,METHYLENE BLUE
5,69,Human,00050474,METHYLENE BLEU LIQ 2%,METHYLENE BLUE
6,115,Human,00037818,BACTERIOSTATIC SODIUM CHLORIDE INJECTION USP,SODIUM CHLORIDE
7,117,Human,00038202,BACTERIOSTATIC WATER FOR INJECTION USP,WATER
8,160,Human,00009881,PAPAVERINE HYDROCHLORIDE INJECTION USP,PAPAVERINE HYDROCHLORIDE
9,193,Human,00013609,GRAVOL SUPPOSITORIES,DIMENHYDRINATE


------

In [48]:
columns_form = ["DRUG_CODE", "_", "PHARMACEUTICAL_FORM", "_"]
df_form = txt_into_df(form, columns_form)
print(len(df_form) == df_form["DRUG_CODE"].nunique())
df_form_combined = df_form.groupby('DRUG_CODE')['PHARMACEUTICAL_FORM'].apply(
    lambda x: '; '.join(x.astype(str).unique())
).reset_index()
print(len(df_form_combined) == df_form_combined["DRUG_CODE"].nunique())
df_form_combined.head(5)

False
True


,DRUG_CODE,PHARMACEUTICAL_FORM
0,10000,POWDER FOR SOLUTION
1,100000,LOTION
2,100001,LOTION
3,100002,SOLUTION
4,10001,DRUG PREMIX


# Merge

In [51]:
df_merged_b = pd.merge(df_merged_a, df_form_combined, on="DRUG_CODE", how="inner")
print(f"After merging - Total rows: {len(df_merged_b)}")
df_merged_b.head(10)

After merging - Total rows: 11523


,DRUG_CODE,CLASS,DRUG_ID,BRAND_NAME,INGREDIENT,PHARMACEUTICAL_FORM
0,9,Human,00015741,TAPAZOLE,METHIMAZOLE,TABLET
1,15,Human,00015229,AVENTYL,NORTRIPTYLINE (NORTRIPTYLINE HYDROCHLORIDE),CAPSULE
2,16,Human,00015237,AVENTYL,NORTRIPTYLINE (NORTRIPTYLINE HYDROCHLORIDE),CAPSULE
3,57,Human,00050520,MINERALE LEGERE HUILE,MINERAL OIL LIGHT,LIQUID
4,68,Human,00050466,METHYLENE BLEU LIQ 1%,METHYLENE BLUE,LIQUID
5,69,Human,00050474,METHYLENE BLEU LIQ 2%,METHYLENE BLUE,LIQUID
6,115,Human,00037818,BACTERIOSTATIC SODIUM CHLORIDE INJECTION USP,SODIUM CHLORIDE,SOLUTION
7,117,Human,00038202,BACTERIOSTATIC WATER FOR INJECTION USP,WATER,SOLUTION
8,160,Human,00009881,PAPAVERINE HYDROCHLORIDE INJECTION USP,PAPAVERINE HYDROCHLORIDE,LIQUID
9,193,Human,00013609,GRAVOL SUPPOSITORIES,DIMENHYDRINATE,SUPPOSITORY


----

# Route

In [17]:
columns_route = ["DRUG_CODE", "_", "ROUTE_OF_ADMINISTRATION", "_"]
df_route = txt_into_df(route, columns_route)
df_route.head()

,DRUG_CODE,ROUTE_OF_ADMINISTRATION
0,9,ORAL
1,15,ORAL
2,16,ORAL
3,57,TOPICAL
4,68,ORAL


In [53]:
df_route_combined = df_route.groupby('DRUG_CODE')['ROUTE_OF_ADMINISTRATION'].apply(
    lambda x: '; '.join(x.astype(str).unique())
).reset_index()
print(len(df_route_combined) == df_route_combined["DRUG_CODE"].nunique())
df_route_combined.head(5)

True


,DRUG_CODE,ROUTE_OF_ADMINISTRATION
0,10000,ORAL
1,100000,TOPICAL
2,100001,TOPICAL
3,100002,INTRAVENOUS
4,10001,ORAL


# Merge

In [54]:
df_merged_c = pd.merge(df_merged_b, df_route_combined, on="DRUG_CODE", how="inner")
print(f"After merging - Total rows: {len(df_merged_c)}")
df_merged_c.head(10)

After merging - Total rows: 11523


,DRUG_CODE,CLASS,DRUG_ID,BRAND_NAME,INGREDIENT,PHARMACEUTICAL_FORM,ROUTE_OF_ADMINISTRATION
0,9,Human,00015741,TAPAZOLE,METHIMAZOLE,TABLET,ORAL
1,15,Human,00015229,AVENTYL,NORTRIPTYLINE (NORTRIPTYLINE HYDROCHLORIDE),CAPSULE,ORAL
2,16,Human,00015237,AVENTYL,NORTRIPTYLINE (NORTRIPTYLINE HYDROCHLORIDE),CAPSULE,ORAL
3,57,Human,00050520,MINERALE LEGERE HUILE,MINERAL OIL LIGHT,LIQUID,TOPICAL
4,68,Human,00050466,METHYLENE BLEU LIQ 1%,METHYLENE BLUE,LIQUID,ORAL
5,69,Human,00050474,METHYLENE BLEU LIQ 2%,METHYLENE BLUE,LIQUID,ORAL
6,115,Human,00037818,BACTERIOSTATIC SODIUM CHLORIDE INJECTION USP,SODIUM CHLORIDE,SOLUTION,INTRAVENOUS; SUBCUTANEOUS; INTRAMUSCULAR
7,117,Human,00038202,BACTERIOSTATIC WATER FOR INJECTION USP,WATER,SOLUTION,INTRAVENOUS; SUBCUTANEOUS; INTRAMUSCULAR
8,160,Human,00009881,PAPAVERINE HYDROCHLORIDE INJECTION USP,PAPAVERINE HYDROCHLORIDE,LIQUID,INTRAMUSCULAR; INTRAVENOUS; SUBCUTANEOUS
9,193,Human,00013609,GRAVOL SUPPOSITORIES,DIMENHYDRINATE,SUPPOSITORY,RECTAL


----

# Status

In [61]:
columns_status = ["DRUG_CODE", "_", "STATUS", "HISTORY_DATE", "_", "_", "_"]
df_status = txt_into_df(status, columns_status)
df_status.head(10)

,DRUG_CODE,STATUS,HISTORY_DATE
0,9,MARKETED,31-DEC-1951
1,9,MARKETED,16-APR-2001
2,9,APPROVED,24-JAN-2001
3,9,APPROVED,18-JAN-2001
4,9,APPROVED,07-MAR-2025
5,9,MARKETED,07-MAR-2025
6,9,APPROVED,07-MAR-2025
7,15,MARKETED,31-DEC-1965
8,15,APPROVED,04-FEB-2002
9,15,APPROVED,28-JAN-2002


In [64]:
df_status["HISTORY_DATE_dt"] = pd.to_datetime(df_status["HISTORY_DATE"]).dt.strftime("%Y.%m.%d")
df_status.head(8)

C:\Users\hh25g551\AppData\Local\Temp\ipykernel_29412\3616042036.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_status["HISTORY_DATE_dt"] = pd.to_datetime(df_status["HISTORY_DATE"]).dt.strftime("%Y.%m.%d")


,DRUG_CODE,STATUS,HISTORY_DATE,HISTORY_DATE_dt
0,9,MARKETED,31-DEC-1951,1951.12.31
1,9,MARKETED,16-APR-2001,2001.04.16
2,9,APPROVED,24-JAN-2001,2001.01.24
3,9,APPROVED,18-JAN-2001,2001.01.18
4,9,APPROVED,07-MAR-2025,2025.03.07
5,9,MARKETED,07-MAR-2025,2025.03.07
6,9,APPROVED,07-MAR-2025,2025.03.07
7,15,MARKETED,31-DEC-1965,1965.12.31


In [65]:
df_status_sorted = df_status.sort_values(['DRUG_CODE', 'HISTORY_DATE_dt'])
df_status_sorted.head(8)

,DRUG_CODE,STATUS,HISTORY_DATE,HISTORY_DATE_dt
3850,10000,MARKETED,31-DEC-1989,1989.12.31
3851,10000,APPROVED,15-MAY-1997,1997.05.15
3852,10000,APPROVED,28-NOV-2001,2001.11.28
3853,10000,APPROVED,30-NOV-2001,2001.11.30
3855,10000,APPROVED,24-FEB-2004,2004.02.24
3854,10000,APPROVED,02-MAR-2004,2004.03.02
3856,10000,MARKETED,06-APR-2005,2005.04.06
3858,10000,APPROVED,27-JAN-2014,2014.01.27


In [66]:
# Get earliest date and status for each drug code
earliest_status = df_status_sorted.groupby('DRUG_CODE').first().reset_index()
earliest_status = earliest_status[['DRUG_CODE', 'STATUS', 'HISTORY_DATE_dt']].rename(columns={
    'STATUS': 'Decision',
    'HISTORY_DATE_dt': 'Decision_date'
})

# Get latest date and status for each drug code
latest_status = df_status_sorted.groupby('DRUG_CODE').last().reset_index()
latest_status = latest_status[['DRUG_CODE', 'STATUS']].rename(columns={
    'STATUS': 'Current_status'
})

# Merge earliest and latest status information
df_status_summary = pd.merge(earliest_status, latest_status, on='DRUG_CODE', how='outer')

print(f"Status summary shape: {df_status_summary.shape}")
print(f"Unique drug codes: {df_status_summary['DRUG_CODE'].nunique()}")
df_status_summary.head(10)

Status summary shape: (13480, 4)
Unique drug codes: 13480


,DRUG_CODE,Decision,Decision_date,Current_status
0,10000,MARKETED,1989.12.31,MARKETED
1,100000,APPROVED,2021.01.08,MARKETED
2,100001,APPROVED,2021.01.08,MARKETED
3,100002,APPROVED,2021.01.11,MARKETED
4,10001,MARKETED,1989.12.31,MARKETED
5,100012,APPROVED,2021.01.19,MARKETED
6,100015,APPROVED,2021.01.12,MARKETED
7,100018,APPROVED,2021.01.12,MARKETED
8,100019,APPROVED,2021.01.19,MARKETED
9,100020,APPROVED,2021.01.19,MARKETED


In [68]:
df_merged_d = pd.merge(df_merged_c, df_status_summary, on='DRUG_CODE', how='inner')
print(len(df_merged_d) == df_merged_d['DRUG_CODE'].nunique())
df_merged_d.head()


True


,DRUG_CODE,CLASS,DRUG_ID,BRAND_NAME,INGREDIENT,PHARMACEUTICAL_FORM,ROUTE_OF_ADMINISTRATION,Decision,Decision_date,Current_status
0,9,Human,00015741,TAPAZOLE,METHIMAZOLE,TABLET,ORAL,MARKETED,1951.12.31,APPROVED
1,15,Human,00015229,AVENTYL,NORTRIPTYLINE (NORTRIPTYLINE HYDROCHLORIDE),CAPSULE,ORAL,MARKETED,1965.12.31,APPROVED
2,16,Human,00015237,AVENTYL,NORTRIPTYLINE (NORTRIPTYLINE HYDROCHLORIDE),CAPSULE,ORAL,MARKETED,1965.12.31,APPROVED
3,57,Human,00050520,MINERALE LEGERE HUILE,MINERAL OIL LIGHT,LIQUID,TOPICAL,MARKETED,1951.12.31,MARKETED
4,68,Human,00050466,METHYLENE BLEU LIQ 1%,METHYLENE BLUE,LIQUID,ORAL,MARKETED,1951.12.31,MARKETED


# comp

In [73]:
columns_comp = ["DRUG_CODE", "_", "_", "COMPANY_NAME", "_", "_", "_", "_", "_", "_", "_", "_", "_", "_", "_", "_", "_", "_"]
df_comp = txt_into_df(comp, columns_comp)
df_comp.head()

,DRUG_CODE,COMPANY_NAME
0,9,ENDO OPERATIONS LTD.
1,15,AA PHARMA INC
2,16,AA PHARMA INC
3,57,LABORATOIRE ATLAS INC
4,68,LABORATOIRE ATLAS INC


In [86]:
df_merged_e = pd.merge(df_merged_d, df_comp, on='DRUG_CODE', how='inner')
print(len(df_merged_e) == df_merged_e['DRUG_CODE'].nunique())
df_merged_e.head()


True


,DRUG_CODE,CLASS,DRUG_ID,BRAND_NAME,INGREDIENT,PHARMACEUTICAL_FORM,ROUTE_OF_ADMINISTRATION,Decision,Decision_date,Current_status,COMPANY_NAME
0,9,Human,00015741,TAPAZOLE,METHIMAZOLE,TABLET,ORAL,MARKETED,1951.12.31,APPROVED,ENDO OPERATIONS LTD.
1,15,Human,00015229,AVENTYL,NORTRIPTYLINE (NORTRIPTYLINE HYDROCHLORIDE),CAPSULE,ORAL,MARKETED,1965.12.31,APPROVED,AA PHARMA INC
2,16,Human,00015237,AVENTYL,NORTRIPTYLINE (NORTRIPTYLINE HYDROCHLORIDE),CAPSULE,ORAL,MARKETED,1965.12.31,APPROVED,AA PHARMA INC
3,57,Human,00050520,MINERALE LEGERE HUILE,MINERAL OIL LIGHT,LIQUID,TOPICAL,MARKETED,1951.12.31,MARKETED,LABORATOIRE ATLAS INC
4,68,Human,00050466,METHYLENE BLEU LIQ 1%,METHYLENE BLUE,LIQUID,ORAL,MARKETED,1951.12.31,MARKETED,LABORATOIRE ATLAS INC


# Final edits

In [87]:
cols_to_drop = ["CLASS"]
df = df_merged_e.copy()
df_merged_e = df.drop(columns=cols_to_drop)
df_merged_e.head()

,DRUG_CODE,DRUG_ID,BRAND_NAME,INGREDIENT,PHARMACEUTICAL_FORM,ROUTE_OF_ADMINISTRATION,Decision,Decision_date,Current_status,COMPANY_NAME
0,9,00015741,TAPAZOLE,METHIMAZOLE,TABLET,ORAL,MARKETED,1951.12.31,APPROVED,ENDO OPERATIONS LTD.
1,15,00015229,AVENTYL,NORTRIPTYLINE (NORTRIPTYLINE HYDROCHLORIDE),CAPSULE,ORAL,MARKETED,1965.12.31,APPROVED,AA PHARMA INC
2,16,00015237,AVENTYL,NORTRIPTYLINE (NORTRIPTYLINE HYDROCHLORIDE),CAPSULE,ORAL,MARKETED,1965.12.31,APPROVED,AA PHARMA INC
3,57,00050520,MINERALE LEGERE HUILE,MINERAL OIL LIGHT,LIQUID,TOPICAL,MARKETED,1951.12.31,MARKETED,LABORATOIRE ATLAS INC
4,68,00050466,METHYLENE BLEU LIQ 1%,METHYLENE BLUE,LIQUID,ORAL,MARKETED,1951.12.31,MARKETED,LABORATOIRE ATLAS INC


In [88]:
# rename cols
mapping = {
    "DRUG_CODE": "DRUG_CODE",
    "DRUG_ID": "Marketing_authorisation_number",
    "BRAND_NAME": "Drug_name",
    "INGREDIENT": "Non_proprietary_name",
    "PHARMACEUTICAL_FORM": "Pharmaceutical_form",
    "ROUTE_OF_ADMINISTRATION": "Administration_route",
    "Decision": "Decision",
    "Decision_date": "Decision_date",
    "Current_status": "Current_status",
    "COMPANY_NAME": "Marketing_authorisation_holder"
}

df_merged_f = df_merged_e.rename(columns=mapping)
df_merged_f.head()

,DRUG_CODE,Marketing_authorisation_number,Drug_name,Non_proprietary_name,Pharmaceutical_form,Administration_route,Decision,Decision_date,Current_status,Marketing_authorisation_holder
0,9,00015741,TAPAZOLE,METHIMAZOLE,TABLET,ORAL,MARKETED,1951.12.31,APPROVED,ENDO OPERATIONS LTD.
1,15,00015229,AVENTYL,NORTRIPTYLINE (NORTRIPTYLINE HYDROCHLORIDE),CAPSULE,ORAL,MARKETED,1965.12.31,APPROVED,AA PHARMA INC
2,16,00015237,AVENTYL,NORTRIPTYLINE (NORTRIPTYLINE HYDROCHLORIDE),CAPSULE,ORAL,MARKETED,1965.12.31,APPROVED,AA PHARMA INC
3,57,00050520,MINERALE LEGERE HUILE,MINERAL OIL LIGHT,LIQUID,TOPICAL,MARKETED,1951.12.31,MARKETED,LABORATOIRE ATLAS INC
4,68,00050466,METHYLENE BLEU LIQ 1%,METHYLENE BLUE,LIQUID,ORAL,MARKETED,1951.12.31,MARKETED,LABORATOIRE ATLAS INC


In [89]:
df_merged_f["Decision_date"] = pd.to_datetime(df_merged_f["Decision_date"]).dt.strftime("%d.%m.%Y")
df_merged_f.head()

,DRUG_CODE,Marketing_authorisation_number,Drug_name,Non_proprietary_name,Pharmaceutical_form,Administration_route,Decision,Decision_date,Current_status,Marketing_authorisation_holder
0,9,00015741,TAPAZOLE,METHIMAZOLE,TABLET,ORAL,MARKETED,31.12.1951,APPROVED,ENDO OPERATIONS LTD.
1,15,00015229,AVENTYL,NORTRIPTYLINE (NORTRIPTYLINE HYDROCHLORIDE),CAPSULE,ORAL,MARKETED,31.12.1965,APPROVED,AA PHARMA INC
2,16,00015237,AVENTYL,NORTRIPTYLINE (NORTRIPTYLINE HYDROCHLORIDE),CAPSULE,ORAL,MARKETED,31.12.1965,APPROVED,AA PHARMA INC
3,57,00050520,MINERALE LEGERE HUILE,MINERAL OIL LIGHT,LIQUID,TOPICAL,MARKETED,31.12.1951,MARKETED,LABORATOIRE ATLAS INC
4,68,00050466,METHYLENE BLEU LIQ 1%,METHYLENE BLUE,LIQUID,ORAL,MARKETED,31.12.1951,MARKETED,LABORATOIRE ATLAS INC


In [91]:
df_merged_f["Decision_year"] = pd.to_datetime(df_merged_f["Decision_date"], format="%d.%m.%Y").dt.year
df_merged_f.head()

,DRUG_CODE,Marketing_authorisation_number,Drug_name,Non_proprietary_name,Pharmaceutical_form,Administration_route,Decision,Decision_date,Current_status,Marketing_authorisation_holder,Decision_year
0,9,00015741,TAPAZOLE,METHIMAZOLE,TABLET,ORAL,MARKETED,31.12.1951,APPROVED,ENDO OPERATIONS LTD.,1951
1,15,00015229,AVENTYL,NORTRIPTYLINE (NORTRIPTYLINE HYDROCHLORIDE),CAPSULE,ORAL,MARKETED,31.12.1965,APPROVED,AA PHARMA INC,1965
2,16,00015237,AVENTYL,NORTRIPTYLINE (NORTRIPTYLINE HYDROCHLORIDE),CAPSULE,ORAL,MARKETED,31.12.1965,APPROVED,AA PHARMA INC,1965
3,57,00050520,MINERALE LEGERE HUILE,MINERAL OIL LIGHT,LIQUID,TOPICAL,MARKETED,31.12.1951,MARKETED,LABORATOIRE ATLAS INC,1951
4,68,00050466,METHYLENE BLEU LIQ 1%,METHYLENE BLUE,LIQUID,ORAL,MARKETED,31.12.1951,MARKETED,LABORATOIRE ATLAS INC,1951


In [92]:
import json
df_merged_f.to_csv("./../data/HealthCanada/HEALTHCANADA.csv", index=False, encoding='utf-8')
with open("./../data/HealthCanada/HEALTHCANADA.json", "w", encoding='utf-8') as json_file:
    json.dump(df_merged_f.to_dict(orient="index"), json_file, indent=4)
